# Dynamics Transformer — perturbation × time → per-gene response

Physics-informed gene-transformer trained on **LINCS L1000**, held out **by perturbation**. **Needs a GPU runtime.**

Cells: GPU/clone/deps → physics half-lives → LINCS table → train+eval → **Stage-1 EGF transfer test (beats 0.25?)** → save to Drive.

In [ ]:
# 1) GPU check + clone repo + deps
import torch, subprocess, os, sys
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE  ->  Runtime > Change runtime type > GPU')
if not os.path.isdir('/content/cell'):
    subprocess.run('git clone -b claude/vectorize-gex-propensity-NRqBW https://github.com/nikku03/cell.git /content/cell', shell=True)
os.chdir('/content/cell'); subprocess.run(['git','pull'])
subprocess.run('pip install -q cmapPy h5py pandas openpyxl', shell=True)
print('cwd', os.getcwd())

In [ ]:
# 2) PHYSICS features: measured mRNA + protein half-lives (CC-BY) + the EGF test course
import subprocess, sys
subprocess.run([sys.executable,'colab/fetch_dynamics_data.py'])
subprocess.run([sys.executable,'colab/fetch_timecourse.py'])   # GSE6783 EGF course for the Stage-1 test

In [ ]:
# 3) LINCS L1000 -> training table  (GSE92742 = full 20 GB, ~20k perts, times 1/2/3/4/6/24 h)
import subprocess, sys, os
os.environ['LINCS_GSE']='GSE92742'        # phase 1, full. 'GSE70138' = smaller 5 GB
os.environ['LINCS_MAX_SIGS']='120000'     # raise on A100 for more coverage
r=subprocess.run([sys.executable,'colab/fetch_lincs.py'],capture_output=True,text=True)
print(r.stdout[-1600:]); print('ERR',r.stderr[-500:]) if r.stderr.strip() else None

In [ ]:
# 4) TRAIN the dynamics transformer (held out by perturbation) + in-distribution eval
import subprocess, sys, os
os.environ['DTF_EPOCHS']='15'; os.environ['DTF_DIM']='128'; os.environ['DTF_LAYERS']='3'; os.environ['DTF_BATCH']='256'
r=subprocess.run([sys.executable,'colab/dynamics_transformer.py'],capture_output=True,text=True)
print(r.stdout[-2200:]); print('ERR',r.stderr[-900:]) if r.stderr.strip() else None

In [ ]:
# 5) STAGE-1 TEST: query the model at EGF's minute-timepoints -> does it BEAT 0.25?
import subprocess, sys
r=subprocess.run([sys.executable,'colab/eval_transformer_egf.py'],capture_output=True,text=True)
print(r.stdout[-1500:]); print('ERR',r.stderr[-600:]) if r.stderr.strip() else None

In [ ]:
# 6) save checkpoint + evals to Drive (reuse in a future runtime)
from google.colab import drive; drive.mount('/content/drive')
import shutil, os, json
d='/content/drive/MyDrive/virtual_cell_data/dynamics_transformer'; os.makedirs(d,exist_ok=True)
for f in ['dynamics_transformer.pt','dynamics_transformer_eval.json','eval_transformer_egf.json','lincs_train.npz']:
    p=f'outputs/orphan/{f}'
    if os.path.exists(p): shutil.copy(p,d); print('saved',f, os.path.getsize(p)//1048576,'MB')